# Model Training

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import xgboost as xgb
import torch
import torch.nn as nn

from numpy.random import shuffle
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import root_mean_squared_error, r2_score
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/crypto_processed.csv') # load dataset
df.head()

,open,high,low,close,volume,marketCap,price_range,return,log_return,volatility,...,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,volume_change,marketcap_change,crypto_id,coin_return_mean_7,coin_return_std_7,coin_volume_mean_7
0,-0.160364,-0.159336,-0.160087,-0.159897,-0.230257,-0.196686,-0.088149,1.071823,0.500222,0.155954,...,-0.159377,-0.136483,-0.158434,-0.137818,-0.007240,0.502942,0,-0.630949,1.111732,-0.230411
1,-0.159830,-0.158831,-0.159789,-0.159359,-0.229148,-0.195676,-0.085869,0.990511,0.443548,0.178573,...,-0.159538,-0.139554,-0.158511,-0.137336,-0.005418,1.337128,0,-0.226869,1.113910,-0.229700
2,-0.159336,-0.158342,-0.159145,-0.158674,-0.226972,-0.195116,-0.087058,1.255303,0.518158,0.149003,...,-0.159479,-0.138412,-0.158580,-0.137504,-0.004925,0.582609,0,0.100680,1.171423,-0.229362
3,-0.158594,-0.158361,-0.158796,-0.159092,-0.228848,-0.195427,-0.090579,-0.718335,-0.307585,0.089019,...,-0.159531,-0.139331,-0.158710,-0.138002,-0.007119,-0.334739,0,-0.079411,1.188986,-0.229153
4,-0.159013,-0.158636,-0.159014,-0.159383,-0.230243,-0.195643,-0.091286,-0.540872,-0.223568,0.084332,...,-0.159607,-0.140395,-0.158868,-0.138726,-0.007103,-0.253112,0,-0.121718,1.201056,-0.229102


In [3]:
df.columns

Index(['open', 'high', 'low', 'close', 'volume', 'marketCap', 'price_range',
       'return', 'log_return', 'volatility', 'year', 'month', 'day',
       'dayofweek', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14',
       'rolling_std_14', 'volume_change', 'marketcap_change', 'crypto_id',
       'coin_return_mean_7', 'coin_return_std_7', 'coin_volume_mean_7'],
      dtype='str')

In [4]:
df.sort_values(["year", "month", "day"], inplace=True)

# XGBoost

In [5]:
X = df.drop(columns=['volatility']) # input
y = df['volatility'] # target

split = int(len(df)*0.8) # Split like this for Time related dataset

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [6]:
tscv = TimeSeriesSplit(n_splits=3)

In [7]:
# Base model
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    tree_method='hist',
    random_state=42,
    shuffle=False
)

In [8]:
# params for Random Search CV
param_grid = {
  'n_estimators':[300, 400, 500, 700],
  'learning_rate':[0.01, 0.03, 0.05, 0.1],
  'max_depth':[4, 6, 8],
  'subsample':[0.7, 0.8, 1],
  'colsample_bytree':[0.7, 0.8, 1],
  'gamma':[0, 0.1, 0.2],
  'min_child_weight':[1, 3, 5]
}

In [9]:
random_search = RandomizedSearchCV(
  estimator=xgb_model,
  param_distributions=param_grid,
  n_iter=10,
  cv=tscv,
  scoring='neg_root_mean_squared_error',
  verbose=2,
  n_jobs=-1,
  random_state=42
)

random_search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.7, 0.8, ...], 'gamma': [0, 0.1, ...], 'learning_rate': [0.01, 0.03, ...], 'max_depth': [4, 6, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies tha

In [10]:
print(random_search.best_params_)

{'subsample': 0.8, 'n_estimators': 700, 'min_child_weight': 5, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 0.2, 'colsample_bytree': 0.7}


In [11]:
# Selecting the best xgboost model
best_xgb_model = random_search.best_estimator_

best_xgb_model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.7
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [12]:
y_pred = best_xgb_model.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('RMSE:', rmse)
print('R2:', r2)

RMSE: 0.042838867845716076
R2: 0.6355488431356273


In [13]:
best_xgb_model.save_model("../models/model.json")

# LSTM

In [14]:
df.sort_values(["crypto_id", "year", "month", "day"], inplace=True)

In [15]:
# Create Sequences
# We use 30 previous timesteps to predict next volatility.

SEQ_LEN = 30

X_seq = []
y_seq = []

for i in range(len(X) - SEQ_LEN):
    X_seq.append(X[i:i+SEQ_LEN])
    y_seq.append(y.iloc[i+SEQ_LEN])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

print(X_seq.shape)

(69512, 30, 23)


In [16]:
split = int(len(X_seq) * 0.8)

X_train = X_seq[:split]
X_test = X_seq[split:]

y_train = y_seq[:split]
y_test = y_seq[split:]

In [17]:
# Convert to PyTorch Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

In [18]:
# Define Batch Size
BATCH_SIZE = 64

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [19]:
class LSTMModel(nn.Module):

    def __init__(self, input_size):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=64,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Linear(64,1)

    def forward(self, x):
        out,_ = self.lstm(x)
        out = out[:,-1,:]
        out = self.fc(out)

        return out

In [20]:
input_size = X_train.shape[2]
model = LSTMModel(input_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [21]:
EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0

    for X_batch, y_batch in train_loader:
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss/len(train_loader)}")

Epoch 1/2, Loss: 0.011607729080516278
Epoch 2/2, Loss: 0.011345068133067472


In [22]:
model.eval()

preds = []
true_vals = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        preds.extend(outputs.numpy())
        true_vals.extend(y_batch.numpy())

rmse = root_mean_squared_error(true_vals, preds)
r2 = r2_score(true_vals, preds)

print("RSME:", rmse)
print("R2 Score:", r2)

RSME: 0.07123236240826739
R2 Score: -0.007432176734598617


# XGBoost score
- **RMSE: 0.042838867845716076**
- **R2: 0.6355488431356273**

# LSTM score
- **RSME: 0.07123236240826739**
- **R2 Score: -0.007432176734598617**

**XGBoost worked best, so it is stored**